In [1]:
import sys

sys.path.append("../")
from agents.trainers.ppo_trainer import PPOTrainer
import gymnasium as gym
import torch
from configs.agents.ppo import PPOConfig
from configs.games.cartpole import CartPoleConfig
from stats.stats import StatTracker

env = gym.make("CartPole-v1", render_mode="rgb_array")
config_dict = {
    "model_name": "ppo_smoke_test",
    "executor_type": "local",
    # --- New Modular Architecture ---
    "architecture": {
        "activation": "tanh",
        "kernel_initializer": "orthogonal",
    },
    # --- PPO Specific Hyperparameters ---
    "clip_param": 0.2,
    "discount_factor": 0.99,
    "gae_lambda": 0.95,
    "steps_per_epoch": 512,
    "train_policy_iterations": 4,
    "train_value_iterations": 4,
    "target_kl": 0.02,
    "entropy_coefficient": 0.01,
    "num_minibatches": 4,
    "training_steps": 500,
    # --- New Modular Heads ---
    "policy_head": {
        "neck": {
            "type": "dense",
            "widths": [64],
        },
    },
    "value_head": {
        "neck": {
            "type": "dense",
            "widths": [64],
        },
    },
    # --- Optimization Blocks ---
    "actor_config": {
        "learning_rate": 2.5e-4,
        "adam_epsilon": 1e-7,
        "clipnorm": 0.5,
    },
    "critic_config": {
        "learning_rate": 2.5e-4,
        "adam_epsilon": 1e-7,
        "clipnorm": 0.5,
    },
    # --- New Action Selector Config ---
    "action_selector": {
        "base": {
            "type": "categorical",  # Base selector type (from registry)
        },
        "decorators": [
            {
                "type": "ppo_injector",  # Decorator to inject log_prob & value
                "kwargs": {},
            }
        ],
    },
}
print("PPO Config")
config = PPOConfig(
    config_dict,
    CartPoleConfig(),
)

trainer = PPOTrainer(
    config,
    env,
    torch.device("cpu"),
    "ppo",
    StatTracker("ppo"),
)

trainer.checkpoint_interval = 100
trainer.test_interval = 100
trainer.test_trials = 50

trainer.train()

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


PPO Config
Using         agent_type                    : ppo
Using         agent_type                    : ppo
Using default results_path                  : results
Using         training_steps                : 500
Using default adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using default learning_rate                 : 0.001
Using default clipnorm                      : 0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using         num_minibatches               : 4
Using default training_iterations           : 1
Using         lr_schedule                   : None
Using         lr_schedule                   : {'type': 'constant', 'initial': 0.001}
Using default test_interval                 : 1000
Using default checkpoint_interval           : 1000
Using default minibatch_size                : 64
Using default replay_buffer_size            : 5000
Using default min_rep

Initializing stat 'score' with subkeys None
Initializing stat 'episode_length' with subkeys None
Initializing stat 'test_score' with subkeys ['avg', 'p0']
Initializing stat 'policy_loss' with subkeys None
Initializing stat 'value_loss' with subkeys None
Initializing stat 'policy_entropy' with subkeys None
Initializing stat 'kl_divergence' with subkeys None
Initializing stat 'explained_variance' with subkeys None
Initializing stat 'learner_fps' with subkeys None
Starting PPO training for 500 steps...
Initializing stat 'PPOPolicyLoss' with subkeys None
Initializing stat 'PPOValueLoss' with subkeys None
Initializing stat 'training_step' with subkeys None
Initializing stat 'loss' with subkeys None
Initializing stat 'ppo_entropy' with subkeys None
Step 10
Step 20
Step 30
Step 40
Step 50
Step 60
Step 70
Step 80
Step 90
plotting score
plotting episode_length
plotting PPOPolicyLoss
plotting PPOValueLoss
plotting training_step
plotting loss
plotting ppo_entropy
Saved stats plot to /Users/jonath

In [1]:
import gymnasium as gym
import sys

sys.path.append("../")
from agents.trainers.rainbow_trainer import RainbowTrainer
from configs.games.cartpole import CartPoleConfig
from configs.agents.rainbow_dqn import RainbowConfig
from stats.stats import StatTracker
import torch

# env = ClipReward(AtariPreprocessing(gym.make("MsPacmanNoFrameskip-v4", render_mode="rgb_array"), terminal_on_life_loss=True), -1, 1) # as recommended by the original paper, should already include max pooling
# env = FrameStack(env, 4)
env = gym.make("CartPole-v1")

config_dict = {
    "executor_type": "local",
    "adam_epsilon": 1e-8,
    "learning_rate": 0.001,
    "training_steps": 3000,
    "minibatch_size": 128,
    "transfer_interval": 100,
    "n_step": 3,
    "replay_interval": 4,
    "kernel_initializer": "orthogonal",
    "clipnorm": 10.0,
    "model_name": "rainbow_smoke_test",
    "noisy_sigma": 0.5,
    # --- MISSING ROOT CONFIG PARAMS ---
    "atom_size": 51,
    "v_min": 0.0,
    "v_max": 200.0,  # Curt Park uses 200 for CartPole
    "representation_mode": "categorical",  # CRITICAL: Triggers Bellman Projection
    "weight_decay": 0.0,  # CRITICAL: rainbow_trainer.py crashes without this
    # --- Backbones ---
    "backbone": {
        "type": "dense",
        "widths": [128],
    },
    # --- Head Configuration (New Nested System) ---
    "head": {
        "output_strategy": {
            "type": "c51",
            "num_atoms": 51,
            # Defaults for v_min/v_max are handled automatically
            # or you can specify them here:
            "v_min": 0,
            "v_max": 500,
        },
        # Hidden layers for Dueling Head (restored to 512 for performance)
        "value_hidden_widths": [],
        "advantage_hidden_widths": [],
        # "noisy_sigma": 0.5 (inherits from global)
    },
    # --- PER (Prioritized Experience Replay) ---
    "per_epsilon": 1e-6,
    "per_alpha": 0.2,
    "per_beta": 0.6,
    "action_selector": {
        "base": {
            "type": "argmax",  # Maps to ArgmaxSelector (Greedy)
            "kwargs": {},
        },
    },
}


game_config = CartPoleConfig()
config = RainbowConfig(config_dict, game_config)
trainer = RainbowTrainer(
    config,
    env,
    torch.device("cpu"),
    "rainbow_refactor",
    StatTracker("rainbow_refactor"),
)

trainer.checkpoint_interval = 100
trainer.test_interval = 500

trainer.train()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using         agent_type                    : rainbow
Using         agent_type                    : rainbow
Using default results_path                  : results
Using         training_steps                : 3000
Using         adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using         learning_rate                 : 0.001
Using         clipnorm                      : 10.0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using         weight_decay                  : 0.0
Using default num_minibatches               : 1
Using default training_iterations           : 1
Using         lr_schedule                   : None
Using         lr_schedule                   : {'type': 'constant', 'initial': 0.001}
Using default test_interval                 : 1000
Using default checkpoint_interval           : 1000
Using         minibatch_size                : 128
Using default replay_buffer_size            : 5000
Using default min_r

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.possible_agents to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.possible_agents` for environment variables or `env.get_wrapper_attr('possible_agents')` that will search the reminding wrappers.
  logger.warn(


Initializing stat 'score' with subkeys None
Initializing stat 'episode_length' with subkeys None
Initializing stat 'test_score' with subkeys ['avg', 'p0']
Initializing stat 'loss' with subkeys None
Initializing stat 'learner_fps' with subkeys None
Initializing stat 'actor_fps' with subkeys None
Starting Rainbow training for 3000 steps...
Initializing stat 'num_episodes' with subkeys None
Initializing stat 'C51Loss' with subkeys None
Initializing stat 'indices' with subkeys None
Initializing stat 'training_step' with subkeys None
plotting score
plotting episode_length
plotting loss
plotting actor_fps
plotting num_episodes
plotting C51Loss
plotting indices
plotting training_step
Saved stats plot to /Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/test_notebooks/checkpoints/rainbow_refactor/graphs/rainbow_refactor_stats.png
Saved checkpoint at step 100 to /Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/test_notebooks/checkpoints/rainbow_refactor/step_100
Step 100, Epsil

AttributeError: 'NoneType' object has no attribute 'stop'

In [ ]:
import sys

sys.path.append("../")
import time
import torch
from agents.random_agent import RandomAgent

# from agents.muzero import MuZeroAgent
from agents.trainers.muzero_trainer import MuZeroTrainer
from configs.agents.muzero import MuZeroConfig
from configs.games.tictactoe import TicTacToeConfig
from agents.tictactoe_expert import TicTacToeBestAgent
from modules.models.world_model import WorldModel
from stats.stats import StatTracker
import torch.nn.functional as F

# Ensure we use CPU for fairness/comparibility or GPU if available
device = "cpu"  # or "cuda" if available
print(f"Using device: {device}")

import torch.nn.functional as F
from modules.models.world_model import WorldModel

params = {
    # General Training Params
    "num_simulations": 25,
    "training_steps": 10000,
    "transfer_interval": 1,
    "minibatch_size": 8,
    "replay_buffer_size": 100000,
    "num_workers": 4,
    "num_envs_per_worker": 1,
    "num_puffer_threads": 1,
    "search_backend": "python",
    "search_batch_size": 5,
    # Search & Learning
    "per_alpha": 0.0,
    "per_beta_schedule": {
        "type": "constant",
        "initial": 0.0,
    },
    "temperature_schedule": {
        "type": "stepwise",
        "steps": [5],
        "values": [1.0, 0.0],
    },
    "n_step": 10,
    "dirichlet_alpha": 0.25,
    "known_bounds": [-1, 1],
    # Action Selector
    "action_selector": {
        "base": {"type": "categorical", "kwargs": {"exploration": True}}
    },
    # --- THE 3 PHASES OF ARCHITECTURE ---
    # 1. Representation Backbone (Observation -> Latent)
    "representation_backbone": {
        "type": "resnet",
        "norm_type": "batch",
        "activation": "relu",
        "filters": [24, 24, 24],
        "kernel_sizes": [3, 3, 3],
        "strides": [1, 1, 1],
    },
    # 2. Dynamics Backbone (Latent + Action -> Latent)
    "dynamics_backbone": {
        "type": "resnet",
        "norm_type": "batch",
        "activation": "relu",
        "filters": [24, 24, 24],
        "kernel_sizes": [3, 3, 3],
        "strides": [1, 1, 1],
    },
    # 3. Prediction Backbone (Latent -> Behavior Tower)
    # Applied to latents immediately before feeding specialized policy/value heads
    "prediction_backbone": {
        "type": "resnet",  # Alternatively: "mlp" with "widths": [128, 128]
        "norm_type": "batch",
        "activation": "relu",
        "filters": [24],
        "kernel_sizes": [3],
        "strides": [1],
    },
    # Behavior Heads (Attached after Prediction Backbone)
    "value_head": {
        # The neck processes the prediction output (spatial) and automatically
        # flattens it before the scalar projection
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "policy_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        # output_strategy defaults to discrete categorical
    },
    # Environment Heads (Attached directly to World Model, bypasses Prediction Tower)
    "reward_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},  # Or {"type": "muzero"} if atom_size > 1
    },
    "to_play_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
    },
    # Advanced / Stochastic MuZero Settings
    "stochastic": False,
    "gumbel": False,
    "gumbel_m": 16,
    "use_virtual_mean": True,
    "virtual_loss": 3.0,
    "atom_size": 1,
    # Class References
    "world_model_cls": WorldModel,
    "policy_loss_function": F.cross_entropy,
}

game_config = TicTacToeConfig()

print("--- Running MuZero Comaprison of Changes ---")
params_batched = params.copy()
params_batched["search_batch_size"] = 5
params_batched["use_virtual_mean"] = True

env_batch = TicTacToeConfig().env_factory()
config_batch = MuZeroConfig(config_dict=params_batched, game_config=game_config)
config_batch.search_batch_size = 5  # Explicitly set

trainer = MuZeroTrainer(
    config_batch,
    env_batch,
    torch.device("cpu"),
    name="muzero_search_refactor",
    stats=StatTracker(name="muzero_search_refactor"),
    test_agents=[RandomAgent(), TicTacToeBestAgent()],
)
trainer.checkpoint_interval = 100
trainer.test_interval = 1000
trainer.test_trials = 100

start_time = time.time()
trainer.train()
end_time = time.time()
print(f"MuZero Batched Time: {end_time - start_time:.2f}s")

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using device: cpu
--- Running MuZero Comaprison of Changes ---
Using         agent_type                    : muzero
Using         agent_type                    : muzero
Using default results_path                  : results
Using         training_steps                : 10000
Using default adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using default learning_rate                 : 0.001
Using default clipnorm                      : 0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using default num_minibatches               : 1
Using default training_iterations           : 1
Using         lr_schedule                   : None
Using         lr_schedule                   : {'type': 'constant', 'initial': 0.001}
Using default test_interval                 : 1000
Using default checkpoint_interval           : 1000
Using         minibatch_size                : 8
Using        

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report 

Initializing stat 'steps_per_second' with subkeys None
Initializing stat 'mean_is_weight' with subkeys None
Initializing stat 'max_is_weight' with subkeys None
Initializing stat 'min_is_weight' with subkeys None
Initializing stat 'ValueLoss_unweighted' with subkeys None
Initializing stat 'ValueLoss' with subkeys None
Initializing stat 'approx_kl' with subkeys None
Initializing stat 'PolicyLoss_unweighted' with subkeys None
Initializing stat 'PolicyLoss' with subkeys None
Initializing stat 'RewardLoss_unweighted' with subkeys None
Initializing stat 'RewardLoss' with subkeys None
Initializing stat 'ToPlayLoss_unweighted' with subkeys None
Initializing stat 'ToPlayLoss' with subkeys None
Initializing stat 'state_value/mean' with subkeys None
Initializing stat 'policy_logits/entropy' with subkeys None
Initializing stat 'loss_pipeline_latency_ms' with subkeys None
Initializing stat 'learner_throughput' with subkeys None
plotting score
  subkey p0
  subkey p1
  subkey avg
plotting loss
plott

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


plotting score
  subkey p0
  subkey p1
  subkey avg
plotting loss
plotting episode_length
plotting steps_per_second
plotting mean_is_weight
plotting max_is_weight
plotting min_is_weight
plotting ValueLoss_unweighted
plotting ValueLoss
plotting approx_kl
plotting PolicyLoss_unweighted
plotting PolicyLoss
plotting RewardLoss_unweighted
plotting RewardLoss
plotting ToPlayLoss_unweighted
plotting ToPlayLoss
plotting state_value/mean
plotting policy_logits/entropy
plotting loss_pipeline_latency_ms
plotting learner_throughput
Saved stats plot to /Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/examples/checkpoints/muzero_search_refactor/graphs/muzero_search_refactor_stats.png
Saved checkpoint at step 1100 to /Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/examples/checkpoints/muzero_search_refactor/step_1100
Step 1100
plotting score
  subkey p0
  subkey p1
  subkey avg
plotting loss
plotting episode_length
plotting steps_per_second
plotting mean_is_weight
plotting max_is_w

: 

In [1]:
import sys

sys.path.append("../")
import time
import torch
from agents.random_agent import RandomAgent

# from agents.muzero import MuZeroAgent
from agents.trainers.muzero_trainer import MuZeroTrainer
from configs.agents.muzero import MuZeroConfig
from configs.games.tictactoe import TicTacToeConfig
from agents.tictactoe_expert import TicTacToeBestAgent
from modules.world_models.muzero_world_model import MuzeroWorldModel
from stats.stats import StatTracker
import torch.nn.functional as F

# Ensure we use CPU for fairness/comparibility or GPU if available
device = "cpu"  # or "cuda" if available
print(f"Using device: {device}")

params = {
    # General Training Params
    "num_simulations": 25,
    "training_steps": 10000,
    "transfer_interval": 1,
    "minibatch_size": 8,
    "replay_buffer_size": 100000,
    "num_workers": 4,
    "num_envs_per_worker": 1,
    "num_puffer_threads": 1,
    "search_backend": "aos",  # <--- Changed search_backend to backend
    # Search & Learning
    "per_alpha": 0.0,
    "per_beta_schedule": {
        "type": "constant",
        "initial": 0.0,
    },
    # If you had temperature scheduling (MuZero typically does), add:
    "temperature_schedule": {
        "type": "stepwise",
        "steps": [5],  # after 5 moves in episode
        "values": [1.0, 0.0],  # start at 1.0, switch to 0.0 after step 5
    },
    "n_step": 10,
    "dirichlet_alpha": 0.25,  # <--- Changed root_dirichlet_alpha to dirichlet_alpha
    "known_bounds": [-1, 1],
    # Action Selector (Required by AgentConfig)
    # Note: MuZeroTrainer currently manually instantiates MCTSDecorator wrapping this,
    # but we define the base strategy here for validation.
    "action_selector": {
        "base": {"type": "categorical", "kwargs": {"exploration": True}}
    },
    # Architecture
    "backbone": {
        "type": "resnet",
        "norm_type": "batch",
        "activation": "relu",
        "filters": [24],
        "kernel_sizes": [3],
        "strides": [1],
        "residual_layers": [(24, 3, 1)],
    },
    # Specialized Heads
    "reward_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "value_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "policy_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        # output_strategy defaults to categorical
    },
    "to_play_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
    },
    # Advanced / Stochastic MuZero
    "stochastic": False,
    "gumbel": True,
    "gumbel_m": 8,
    "search_batch_size": 0,
    "use_virtual_mean": True,
    "virtual_loss": 3.0,
    "atom_size": 1,
    # Class References
    "world_model_cls": MuzeroWorldModel,
    "policy_loss_function": F.kl_div,  # kl_div <--- Changed policy_loss to policy_loss_function
}
game_config = TicTacToeConfig()

print("--- Running MuZero Comaprison of Changes ---")
params_batched = params.copy()
env_batch = TicTacToeConfig().make_env()
config_batch = MuZeroConfig(config_dict=params_batched, game_config=game_config)
config_batch.search_batch_size = 5  # Explicitly set

trainer = MuZeroTrainer(
    config_batch,
    env_batch,
    torch.device("cpu"),
    name="muzero_search_refactor",
    stats=StatTracker(name="muzero_search_gumbel"),
    test_agents=[RandomAgent(), TicTacToeBestAgent()],
)
trainer.checkpoint_interval = 100
trainer.test_interval = 1000
trainer.test_trials = 100

start_time = time.time()
trainer.train()
end_time = time.time()
print(f"MuZero Batched Time: {end_time - start_time:.2f}s")

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


ModuleNotFoundError: No module named 'modules.world_models.muzero_world_model'

In [1]:
import sys

sys.path.append("../")
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
import random

from custom_gym_envs.envs.matching_pennies import (
    env as matching_pennies_env,
    MatchingPenniesGymEnv,
)
from agents.trainers.nfsp_trainer import NFSPTrainer
from agent_configs.dqn.nfsp_config import NFSPDQNConfig
from agent_configs.dqn.rainbow_config import RainbowConfig
from game_configs.matching_pennies_config import MatchingPenniesConfig
from learners.losses.basic_losses import (
    KLDivergenceLoss,
    CategoricalCrossentropyLoss,
    HuberLoss,
    MSELoss,
)
import torch
from torch.optim import Adam, SGD
from stats.stats import StatTracker

config_dict = {
    "shared_networks_and_buffers": False,
    "training_steps": 10000,
    "anticipatory_param": 0.1,
    "replay_interval": 2,
    "num_minibatches": 1,
    "learning_rate": 0.1,
    "momentum": 0.0,
    "optimizer": SGD,
    "loss_function": HuberLoss(),
    "min_replay_buffer_size": 500,
    "minibatch_size": 128,
    "replay_buffer_size": 1000,
    "transfer_interval": 300,
    # --- NEW RL BACKBONE ---
    "backbone": {"type": "dense", "widths": [128]},
    # Note: value_backbone and advantage_backbone default to Identity (None)
    # which replaces the old empty list [] behavior.
    "noisy_sigma": 0.0,
    "eg_epsilon": 0.06,
    "eg_epsilon_decay_type": "inverse_sqrt",
    "eg_epsilon_decay_final_step": 0,
    # --- SL CONFIG ---
    "sl_learning_rate": 0.01,
    "sl_momentum": 0.0,
    "sl_optimizer": SGD,
    "sl_loss_function": CategoricalCrossentropyLoss(),
    "sl_min_replay_buffer_size": 500,
    "sl_minibatch_size": 128,
    "sl_replay_buffer_size": 20000,
    # --- NEW SL BACKBONE ---
    "sl_backbone": {"type": "dense", "widths": [128]},
    "sl_clip_low_prob": 0.0,
    "per_alpha": 0.0,
    "per_beta": 0.0,
    "per_beta_final": 0.0,
    "per_epsilon": 0.00001,
    "n_step": 1,
    "atom_size": 1,
    "dueling": False,
    "clipnorm": 10.0,
    "sl_clipnorm": 10.0,
}

config = NFSPDQNConfig(
    config_dict=config_dict,
    game_config=MatchingPenniesConfig(),
)
config.save_intermediate_weights = True

import custom_gym_envs
import gymnasium as gym

# env = gym.make("custom_gym_envs/LeducHoldem-v0", encode_player_turn=False)

env = matching_pennies_env(render_mode="rgb_array", max_cycles=1)

trainer = NFSPTrainer(config, env, torch.device("cpu"), StatTracker("nfsp_3"))
trainer.checkpoint_interval = 100
trainer.test_interval = 1000
trainer.test_trials = 10000
trainer.train()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


ModuleNotFoundError: No module named 'agent_configs'

In [ ]:
import sys

sys.path.append("../")
import torch
import gymnasium as gym
import numpy as np
from configs.games.game import GameConfig
from configs.agents.ppo import PPOConfig
from agents.trainers.ppo_trainer import PPOTrainer
from stats.stats import StatTracker


class ContinuousActionWrapper(gym.ActionWrapper):
    def action(self, action):
        # Pendulum expects an array-like of shape (1,)
        if np.isscalar(action):
            return np.array([action], dtype=np.float32)
        return np.asarray(action, dtype=np.float32).reshape(self.action_space.shape)


def make_pendulum_env(render_mode=None):
    env = gym.make("Pendulum-v1", render_mode=render_mode)
    return ContinuousActionWrapper(env)


class ContinuousControlGameConfig(GameConfig):
    def __init__(self):
        super().__init__(
            num_actions=None,
            max_score=0.0,
            min_score=-2000.0,
            is_discrete=False,
            is_image=False,
            is_deterministic=False,
            has_legal_moves=False,
            perfect_information=True,
            multi_agent=False,
            num_players=1,
            make_env=make_pendulum_env,
        )


game_config = ContinuousControlGameConfig()
config = PPOConfig(
    {
        "model_name": "ppo_continuous_smoke_notebook",
        "multi_process": False,
        "num_workers": 1,
        "training_steps": 100000,
        "steps_per_epoch": 1024,
        "train_policy_iterations": 4,
        "train_value_iterations": 4,
        "num_minibatches": 4,
        "clip_param": 0.2,
        "discount_factor": 0.99,
        "gae_lambda": 0.95,
        "target_kl": 0.02,
        "entropy_coefficient": 0.1,
        "architecture": {
            "activation": "tanh",
            "kernel_initializer": "orthogonal",
        },
        "policy_head": {
            "neck": {
                "type": "dense",
                "widths": [32],
            },
            "output_strategy": {
                "type": "gaussian",
                "action_dim": 1,
                "min_log_std": -5.0,
                "max_log_std": -2.0,
            },
        },
        "value_head": {
            "neck": {
                "type": "dense",
                "widths": [32],
            },
        },
        "actor_config": {
            "learning_rate": 1e-3,
            "adam_epsilon": 1e-7,
            "clipnorm": 1.0,
        },
        "critic_config": {
            "learning_rate": 1e-3,
            "adam_epsilon": 1e-7,
            "clipnorm": 1.0,
        },
        "action_selector": {
            "base": {
                "type": "categorical",
            },
            "decorators": [
                {
                    "type": "ppo_injector",
                    "kwargs": {},
                }
            ],
        },
    },
    game_config,
)

trainer = PPOTrainer(
    config,
    game_config.make_env(),
    torch.device("cpu"),
    stats=StatTracker("ppo_continuous_notebook"),
)

# Avoid writing checkpoints during notebook smoke tests.
trainer.checkpoint_interval = 1000
trainer.test_interval = 1000
trainer.test_trials = 100

trainer.train()